In [1]:
import pathlib, sys
import matplotlib
matplotlib.use('Agg')
_root = pathlib.Path.cwd()
if not (_root / 'src').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# 03 · Old Models vs New Models

The single idea the whole project turns on: the *same* cleaning, applied to a classic TF-IDF model and to a modern transformer, moves them in **different** directions. TF-IDF was built for a sparse bag-of-words world where cleaning helps; the transformer was pretrained on raw text, so cleaning throws away signal it learned to use.

In [2]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from src.cleaning.pipelines import PIPELINE_NAMES
from src.config import get_settings
from src.datasets.corpora import load_sentiment
from src.evaluation.classification import evaluate_classification
from src.models.bow import BowClassifier
from src.models.transformer import TransformerEmbedder
from src.visualisation import plot_pipeline_scores, save_figure, save_json
seed = get_settings().seed
embedder = TransformerEmbedder()
texts, labels = load_sentiment()

## A cross-validated score for the transformer

Encode once per pipeline (cached to `data/emb/`), then cross-validate a logistic head on the embeddings — the same evaluation the TF-IDF model gets.

In [3]:
def transformer_cv(texts, labels, pipeline, n_splits=5):
    emb = embedder.encode(texts, pipeline=pipeline)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    clf = LogisticRegression(max_iter=1000, random_state=seed)
    acc = cross_val_score(clf, emb, labels, cv=skf, scoring='accuracy').mean()
    f1 = cross_val_score(clf, emb, labels, cv=skf, scoring='f1_macro').mean()
    return float(acc), float(f1)

rows = []
for p in PIPELINE_NAMES:
    r = evaluate_classification(BowClassifier, texts, labels, pipeline=p)
    rows.append({'model': 'tfidf+logreg', 'pipeline': p, 'accuracy': r.accuracy, 'macro_f1': r.macro_f1})
    acc, f1 = transformer_cv(texts, labels, p)
    rows.append({'model': 'transformer', 'pipeline': p, 'accuracy': acc, 'macro_f1': f1})
results = pd.DataFrame(rows)
save_json(results, 'nb03_old_vs_new.json')
results.round(3)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,model,pipeline,accuracy,macro_f1
0,tfidf+logreg,none,0.988,0.987
1,transformer,none,1.000,1.000
2,tfidf+logreg,minimal,0.988,0.987
3,transformer,minimal,1.000,1.000
4,tfidf+logreg,standard,0.975,0.975
5,transformer,standard,0.988,0.987
6,tfidf+logreg,aggressive,0.962,0.962
7,transformer,aggressive,0.962,0.962


In [4]:
fig = plot_pipeline_scores(results, metric='accuracy')
fig.axes[0].set_title('Accuracy by pipeline: classic vs modern model (sentiment)')
save_figure(fig, 'nb03_old_vs_new_accuracy.png')

## The direction of the effect

For each model, how much does cleaning move accuracy away from raw text? A negative number means cleaning *hurt* the model.

In [5]:
pivot = results.pivot_table(index='model', columns='pipeline', values='accuracy')
effect = pd.DataFrame({
    'model': pivot.index,
    'raw(none)': pivot['none'].values,
    'standard': pivot['standard'].values,
    'aggressive': pivot['aggressive'].values,
    'standard_minus_raw': (pivot['standard'] - pivot['none']).values,
    'aggressive_minus_raw': (pivot['aggressive'] - pivot['none']).values,
}).reset_index(drop=True)
save_json(effect, 'nb03_cleaning_effect.json')
effect.round(3)

,model,raw(none),standard,aggressive,standard_minus_raw,aggressive_minus_raw
0,tfidf+logreg,0.988,0.975,0.962,-0.013,-0.025
1,transformer,1.000,0.988,0.962,-0.012,-0.037


In [6]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4.5))
x = range(len(effect))
ax.bar([i - 0.2 for i in x], effect['standard_minus_raw'], width=0.4, label='standard - raw', color='#4c72b0')
ax.bar([i + 0.2 for i in x], effect['aggressive_minus_raw'], width=0.4, label='aggressive - raw', color='#c44e52')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(list(x)); ax.set_xticklabels(effect['model'])
ax.set_ylabel('accuracy change vs raw text'); ax.legend()
ax.set_title('What cleaning does to each model (below zero = cleaning hurts)')
fig.tight_layout()
save_figure(fig, 'nb03_cleaning_effect.png')

## Takeaway

Read the effect table: cleaning barely helps — and often hurts — the classic model on this data, while it costs the transformer the most under the aggressive pipeline. The mechanism is simple: subword tokenizers already handle word forms, so stemming is redundant and destructive, and the model learned from the casing and stopwords that cleaning deletes. Feeding a transformer cleaned text hands it a degraded version of exactly what it was pretrained to consume.